# What 10 English-speaking markets autocomplete

Every keyword tool offers country targeting. Hardly any of them tell you whether it is worth using. This notebook answers that with numbers.

**Collected with the [Long Tail Keyword Questions actor](https://apify.com/bhole_krishna/keyword-demand-scraper) on Apify.** Every row here is a live suggestion from Google, YouTube, Bing or DuckDuckGo - the list that drops down while you type - pulled through that actor. The overlap figure is different for every topic, so the only way to know yours is to collect it: **apify.com/bhole_krishna/keyword-demand-scraper**

---

Four questions this notebook answers:

1. If you run the same seed in ten countries, how different are the results?
2. What does a market-exclusive keyword actually look like?
3. Is one search engine enough?
4. Do some markets ask more questions than others?

The answer to the first one is the reason this dataset exists, and it is not the answer most keyword tools imply.

In [ ]:
import glob

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

BLUE, ORANGE, GREEN = "#2a78d6", "#eb6834", "#1baf7a"
MARKETS = ["US", "GB", "IN", "AE", "CA", "AU", "PH", "NG", "ZA", "SG"]

# Found by name rather than by a hard-coded dataset slug, so this notebook keeps
# working if the dataset is renamed or attached alongside others.
def find(filename):
    hits = (glob.glob(f"/kaggle/input/**/{filename}", recursive=True)
            or glob.glob(f"**/{filename}", recursive=True))
    if not hits:
        raise FileNotFoundError(f"{filename} not found - is the dataset attached?")
    return hits[0]

suggestions = pd.read_csv(find("suggestions.csv"))
matrix = pd.read_csv(find("keyword_market_matrix.csv"))
pairs = pd.read_csv(find("market_overlap_pairs.csv"))
engines = pd.read_csv(find("engine_contribution.csv"))

print(f"{len(suggestions):,} suggestions, {len(matrix):,} distinct keywords, "
      f"{len(pairs)} market pairs")
matrix.head()

## 1. How different are ten markets, really?

Two ways to measure it. **Overlap** is how much any two markets share. **Spread** is how many of the ten markets offered each keyword - a keyword in all ten is universal, a keyword in one is local.

In [ ]:
seeds = ["credit card", "running shoes", "web hosting"]

summary = []
for seed in seeds:
    m = matrix[matrix["seed"] == seed]
    total = len(m)
    summary.append({
        "seed": seed,
        "keywords": total,
        "in all 10 markets %": round(100 * (m["market_count"] == 10).sum() / total, 1),
        "in only 1 market %": round(100 * (m["market_count"] == 1).sum() / total, 1),
        "median pair overlap %": round(
            pairs[pairs["seed"] == seed]["overlap_pct_jaccard"].median(), 1),
    })
summary = pd.DataFrame(summary).set_index("seed")
display(summary)

middle = 100 - summary["in all 10 markets %"] - summary["in only 1 market %"]
layers = [(summary["in all 10 markets %"], GREEN, "in all 10 markets"),
          (middle, BLUE, "in 2-9 markets"),
          (summary["in only 1 market %"], ORANGE, "in only 1 market")]

fig, ax = plt.subplots(figsize=(10, 3.4))
left = pd.Series(0.0, index=summary.index)
for values, colour, label in layers:
    ax.barh(summary.index, values, left=left, color=colour, height=0.6, label=label)
    for y, (v, l) in enumerate(zip(values, left)):
        if v > 7:
            ax.text(l + v / 2, y, f"{v:.0f}%", ha="center", va="center",
                    color="white", fontweight="bold")
    left = left + values

ax.set_title("Whether keywords differ by country is decided by the topic",
             fontsize=13, fontweight="bold", loc="left")
ax.set_xlim(0, 100)
ax.xaxis.set_major_formatter(mticker.PercentFormatter())
ax.spines[["top", "right", "left"]].set_visible(False)
ax.legend(frameon=False, ncol=3, loc="upper center", bbox_to_anchor=(0.5, -0.12))
plt.tight_layout()
plt.show()

gap = summary["median pair overlap %"].max() - summary["median pair overlap %"].min()
print("Credit cards are issued by local banks under local rules. "
      "Web hosting is the same product everywhere.")
print(f"That difference is worth {gap:.0f} percentage points of overlap.")

## 2. What a market-exclusive keyword looks like

The percentages stay abstract until you read the rows. These are keywords that exactly one of the ten markets suggested.

In [ ]:
exclusive = matrix[matrix["market_count"] == 1]

counts = (exclusive.groupby(["seed", "markets"]).size()
          .unstack(fill_value=0).reindex(columns=MARKETS, fill_value=0))
print("Market-exclusive keywords per market:")
display(counts)

for seed in ["credit card", "running shoes"]:
    print(f"\n--- {seed} ---")
    for market in ["IN", "ZA", "SG", "PH", "AE", "NG"]:
        rows = exclusive[(exclusive["seed"] == seed) & (exclusive["markets"] == market)]
        examples = rows.sort_values("best_rank")["keyword"].head(3).tolist()
        if examples:
            print(f"  {market}: " + "; ".join(examples))

print("\nLocal banks, local retailers, local languages, local cities. No US-centric keyword")
print("database contains these, because they are not US demand.\n")
print("Now notice GB - British and American search language for these topics is nearly")
print("the same list:")
print(counts["GB"].to_string())

## 3. Is one engine enough?

Most keyword tools read Google and stop. This dataset put the same queries through four engines in the same ten markets, so the question can be answered rather than argued about.

In [ ]:
overall = engines[engines["scope"] == "ALL SEEDS"]
order = ["google", "youtube", "bing", "duckduckgo"]
real = overall[overall["engine"].isin(order)].set_index("engine").loc[order]
display(real[["keywords_found", "share_of_all_pct", "exclusive_to_this_engine",
              "exclusive_share_of_own_pct"]])

total = int(overall["keywords_in_scope"].iloc[0])
excl = 100 * real["exclusive_to_this_engine"] / total
shared = real["share_of_all_pct"] - excl

fig, ax = plt.subplots(figsize=(10, 3.2))
ax.barh(order[::-1], excl[::-1], color=ORANGE, height=0.6,
        label="found by this engine alone")
ax.barh(order[::-1], shared[::-1], left=excl[::-1], color=BLUE, height=0.6,
        label="also found elsewhere")
for y, name in enumerate(order[::-1]):
    ax.text(real.loc[name, "share_of_all_pct"] + 1, y,
            f"{real.loc[name, 'share_of_all_pct']:.1f}%", va="center", color="#52514e")

ax.set_title(f"Share of all {total:,} keywords each engine found", fontsize=13,
             fontweight="bold", loc="left")
ax.set_xlim(0, 70)
ax.xaxis.set_major_formatter(mticker.PercentFormatter())
ax.spines[["top", "right", "left"]].set_visible(False)
ax.legend(frameon=False, ncol=2, loc="upper center", bbox_to_anchor=(0.5, -0.12))
plt.tight_layout()
plt.show()

missed = overall[overall["engine"] == "google-only tool would miss"]
print(f"A Google-only tool misses {missed['share_of_all_pct'].iloc[0]}% of these keywords "
      f"({int(missed['keywords_found'].iloc[0]):,} of {total:,}).")
print("YouTube is close to a separate corpus: more than two thirds of what it suggests, "
      "no other engine does.")

## 4. Do some markets ask more questions than others?

Every keyword carries a rule-based intent label. The `question` ones start with how, what, why, when, where, which, who, can, is, are, does or will - the ones that map straight onto a blog post or an FAQ answer.

In [ ]:
by_market = (suggestions.drop_duplicates(["seed", "market", "keyword"])
             .groupby("market")["intent"]
             .value_counts(normalize=True).unstack(fill_value=0) * 100)
by_market = by_market.reindex(MARKETS)

fig, ax = plt.subplots(figsize=(10, 3.6))
ax.bar(by_market.index, by_market["question"], color=BLUE, width=0.6)
ax.axhline(by_market["question"].mean(), color=ORANGE, linestyle="--", linewidth=1.4,
           label=f"mean {by_market['question'].mean():.1f}%")
ax.set_title("Share of each market's keywords that are questions", fontsize=13,
             fontweight="bold", loc="left")
ax.yaxis.set_major_formatter(mticker.PercentFormatter())
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

display(by_market.round(1))
q = by_market["question"]
spread = q.max() - q.min()
print(f"High: {q.idxmax()} at {q.max():.1f}%   Low: {q.idxmin()} at {q.min():.1f}%   "
      f"Spread: {spread:.1f} points")

if spread < 5:
    print()
    print("So: no. All ten markets ask questions at very nearly the same rate.")
    print("The markets differ in WHAT they ask, not in how often they ask a question -")
    print("which is worth knowing, because it means the question share of a run is stable")
    print("and you can plan FAQ content the same way in every market.")

## Where the data came from, and how to extend it

Every row was collected from the four engines' own public suggestion endpoints with the [**Long Tail Keyword Questions actor**](https://apify.com/bhole_krishna/keyword-demand-scraper) on Apify. No login, no API key, no paid keyword database.

The headline finding is that **the overlap figure is a property of your topic**, so the number in this notebook is not your number. One run per market gives you yours:

```json
{
  "seeds": ["your seed term"],
  "engines": ["google", "youtube", "bing", "duckduckgo"],
  "language": "en",
  "country": "US",
  "includeLetters": true,
  "includeQuestions": true
}
```

The actor also labels every keyword with search intent, records its rank in each engine's suggestion list, and where a keyword clearly names a real topic it attaches Wikipedia's counted monthly pageviews instead of a modelled search-volume estimate.

### Three caveats worth respecting

**Three seeds is a demonstration, not a law.** They were chosen to span local-service, physical-product and global-digital categories. They prove the overlap figure moves enormously by topic; they do not tell you what it is for yours.

**Language was held fixed at `en` and only the country changed.** Adding Germany or Brazil would have mixed language differences into market differences, and "German keywords differ from English ones" is not a finding.

**One snapshot, 21 November 2026.** Suggestion lists are live and they move.

Full column documentation: [github.com/krishnadubey965/autocomplete-10-markets](https://github.com/krishnadubey965/autocomplete-10-markets)